# turkish_mmlu → TUS Dahiliye Soruları: Sayım, Temizlik ve Test Seti

Bu notebook şunları yapar:

1. `alibayram/turkish_mmlu` veri setini indirir ve **kaç TUS sorusu olduğunu** sayar.
2. TUS soruları içinden **dahiliye (iç hastalıkları) sorularını** yan dallara göre ayıklar.
3. Görsele dayalı, bozuk ve tekrar eden soruları işaretleyip temizler.
4. Sabit tohumla **test (500) ve geliştirme (50) setlerini** ayırıp CSV/JSONL olarak kaydeder.
5. Bonus: TUSGPT eğitim verisindeki tahmini dahiliye oranını ve **test–eğitim çakışmasını** kontrol eder.

**Başlamadan önce:** Hugging Face'e giriş yap → https://huggingface.co/datasets/alibayram/turkish_mmlu sayfasında koşulları kabul et → https://huggingface.co/settings/tokens adresinden *Read* yetkili bir token oluştur.

Colab menüsünden **Çalışma zamanı → Tümünü çalıştır** demen yeterli. GPU gerekmez.

In [ ]:
!pip -q install -U datasets pandas scikit-learn huggingface_hub

## 1. Giriş ve veri setini yükleme

In [ ]:
from huggingface_hub import login
login()  # Açılan kutuya token'ını yapıştır

In [ ]:
import re, json, os, unicodedata
import numpy as np
import pandas as pd
from datasets import load_dataset

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 120)

ds = load_dataset("alibayram/turkish_mmlu")
print(ds)

parcalar = []
for split_adi in ds.keys():
    d = ds[split_adi].to_pandas()
    d["split"] = split_adi
    parcalar.append(d)
df = pd.concat(parcalar, ignore_index=True)

print("\nToplam satır:", len(df))
print("Sütunlar:", list(df.columns))
df.head(3)

## 2. Sütun adlarını otomatik bulma

In [ ]:
def sutun_bul(adaylar):
    kucuk = {c: c.lower() for c in df.columns}
    for c, lc in kucuk.items():
        if lc in adaylar:
            return c
    for c, lc in kucuk.items():
        if any(a in lc for a in adaylar):
            return c
    return None

COL_BOLUM    = sutun_bul(["bolum", "bölüm", "section"])
COL_KONU     = sutun_bul(["konu", "topic", "subject"])
COL_SORU     = sutun_bul(["soru", "question"])
COL_SECENEK  = sutun_bul(["secenekler", "seçenekler", "choices", "options"])
COL_CEVAP    = sutun_bul(["cevap", "answer"])
COL_ACIKLAMA = sutun_bul(["aciklama", "açıklama", "explanation"])

for ad, deger in [("Bölüm", COL_BOLUM), ("Konu", COL_KONU), ("Soru", COL_SORU),
                  ("Seçenekler", COL_SECENEK), ("Cevap", COL_CEVAP), ("Açıklama", COL_ACIKLAMA)]:
    print(f"{ad:<11}: {deger}")

assert COL_BOLUM and COL_SORU, "Bölüm/soru sütunu bulunamadı. Yukarıdaki sütun listesine bakıp elle yaz: COL_BOLUM = '...'"

## 3. Türkçe karakterleri sadeleştiren yardımcı fonksiyonlar

In [ ]:
TR_MAP = str.maketrans({"ı": "i", "İ": "i", "I": "i", "ş": "s", "Ş": "s", "ğ": "g", "Ğ": "g",
                        "ü": "u", "Ü": "u", "ö": "o", "Ö": "o", "ç": "c", "Ç": "c", "â": "a", "î": "i", "û": "u"})

def sade(metin):
    """Küçük harfe çevirir, Türkçe karakterleri ASCII'ye indirger, fazla boşlukları siler."""
    if metin is None or (isinstance(metin, float) and np.isnan(metin)):
        return ""
    m = str(metin).translate(TR_MAP).lower()
    m = unicodedata.normalize("NFKD", m)
    m = "".join(ch for ch in m if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", m).strip()

def secenek_listesi(x):
    if x is None:
        return []
    if isinstance(x, (list, tuple, np.ndarray)):
        return [str(s) for s in x]
    s = str(x).strip()
    try:
        v = json.loads(s)
        if isinstance(v, list):
            return [str(t) for t in v]
    except Exception:
        pass
    try:
        import ast
        v = ast.literal_eval(s)
        if isinstance(v, (list, tuple)):
            return [str(t) for t in v]
    except Exception:
        pass
    return [p.strip() for p in re.split(r"\n|\|", s) if p.strip()]

HARFLER = "ABCDEFGHIJ"

def cevap_harfi(cevap, secenekler):
    """Cevap sıfırdan başlayan indeks, harf veya seçenek metni olabilir; hepsini harfe çevirir."""
    if cevap is None:
        return None
    if isinstance(cevap, (int, np.integer)):
        return HARFLER[int(cevap)] if 0 <= int(cevap) < len(secenekler) else None
    s = str(cevap).strip()
    if s.isdigit():
        i = int(s)
        return HARFLER[i] if 0 <= i < len(secenekler) else None
    if len(s) == 1 and s.upper() in HARFLER[:len(secenekler)]:
        return s.upper()
    for i, sec in enumerate(secenekler):
        if sade(sec) == sade(s):
            return HARFLER[i]
    return None

## 4. Bütün bölümler ve TUS bölümünü bulma

In [ ]:
bolum_say = df[COL_BOLUM].value_counts()
print(f"Toplam {len(bolum_say)} bölüm var.\n")

tus_adaylar = [b for b in bolum_say.index if re.search(r"\btus\b|tipta uzmanlik", sade(b))]
saglik_adaylar = [b for b in bolum_say.index
                  if re.search(r"\btip\b|tip fak|saglik|hemsire|eczaci|dis hekim|\bebe", sade(b))
                  and b not in tus_adaylar]

print("TUS olarak görünen bölümler:")
for b in tus_adaylar:
    print(f"   {b}: {bolum_say[b]:,}")
print("\nSağlıkla ilgili görünen diğer bölümler (kontrol için):")
for b in saglik_adaylar:
    print(f"   {b}: {bolum_say[b]:,}")

bolum_say.to_frame("soru_sayisi")

**Kontrol et:** Yukarıdaki tabloda TUS bölümünün adı farklı yazılmışsa (ör. *Tıpta Uzmanlık*), aşağıdaki `TUS_BOLUMLERI` listesini elle düzelt.

In [ ]:
TUS_BOLUMLERI = tus_adaylar          # Gerekirse elle: ["TUS"]
tus = df[df[COL_BOLUM].isin(TUS_BOLUMLERI)].copy()
print(f"TUS soru sayısı: {len(tus):,}  (tüm verinin %{100 * len(tus) / len(df):.2f}'i)")
print(tus["split"].value_counts().to_string())

if COL_KONU:
    konu_say = tus[COL_KONU].value_counts()
    print(f"\nTUS içinde {len(konu_say)} farklı konu var. İlk 60:")
    display(konu_say.head(60).to_frame("soru_sayisi"))

## 5. Dahiliye sorularını yan dallara göre ayıklama

Kural: **Soru dahiliye yan dallarından birine aitse alınır.** Temel bilimler (fizyoloji, farmakoloji vb.), pediatri ve cerrahi dışarıda bırakılır.
Önce `konu` sütunu kullanılır. Konu alanı branş bilgisi taşımıyorsa soru metnindeki terimlerle **tahmini** etiketleme yapılır.

In [ ]:
YAN_DALLAR = {
    "Kardiyoloji":        ["kardiyo", "kalp hastalik", "kalp yetmez", "aritmi", "koroner"],
    "Nefroloji":          ["nefro", "bobrek hastalik", "bobrek yetmez", "glomerul"],
    "Endokrinoloji":      ["endokrin", "diyabet", "diabet", "tiroid", "adrenal", "hipofiz"],
    "Gastroenteroloji":   ["gastro", "hepato", "karaciger hastalik", "pankreatit", "inflamatuvar bagirsak"],
    "Hematoloji":         ["hematoloji", "anemi", "losemi", "lenfoma", "koagulasyon", "hemostaz"],
    "Onkoloji":           ["onkoloji", "medikal onkoloji"],
    "Romatoloji":         ["romato", "vaskulit", "lupus", "artrit"],
    "Göğüs hastalıkları": ["gogus hastalik", "pulmo", "koah", "astim", "akciger hastalik"],
    "Enfeksiyon":         ["enfeksiyon hastalik", "infeksiyon hastalik", "klinik mikrobiyoloji"],
    "Geriatri":           ["geriatri"],
    "Alerji-İmmünoloji":  ["alerji", "klinik immunoloji"],
    "Genel dahiliye":     ["dahiliye", "ic hastalik"],  # en sonda: önce spesifik yan dal aranır
}
HARIC = ["fizyoloji", "anatomi", "histoloji", "embriyoloji", "biyokimya", "farmakoloji",
         "patoloji", "mikrobiyoloji", "genetik", "pediatri", "cocuk", "cerrahi", "kadin hastalik",
         "dogum", "psikiyatri", "noroloji", "dermatoloji", "goz", "kulak burun", "uroloji", "ortopedi",
         "radyoloji", "anestezi", "halk sagligi", "adli tip"]

def yan_dal_bul(metin, haric_kontrol=True):
    m = sade(metin)
    if not m:
        return None
    if haric_kontrol and any(h in m for h in HARIC):
        # "klinik mikrobiyoloji" gibi istisnalar önce kontrol edilir
        if not any(k in m for k in ["klinik mikrobiyoloji", "klinik immunoloji"]):
            return None
    for dal, anahtarlar in YAN_DALLAR.items():
        if any(a in m for a in anahtarlar):
            return dal
    return None

tus["yan_dal"] = None
tus["etiket_yontemi"] = None

if COL_KONU:
    tus["yan_dal"] = tus[COL_KONU].map(yan_dal_bul)
    tus.loc[tus["yan_dal"].notna(), "etiket_yontemi"] = "konu"
    oran = tus["yan_dal"].notna().mean()
    print(f"Konu alanıyla eşleşen TUS sorusu: {tus['yan_dal'].notna().sum():,} (%{100 * oran:.1f})")

    eslesen = (tus[tus["yan_dal"].notna()]
               .groupby([COL_KONU, "yan_dal"]).size().sort_values(ascending=False)
               .rename("soru_sayisi").reset_index())
    print("\nEşleşen konular (yanlış eşleşme var mı kontrol et):")
    display(eslesen.head(60))

    eslesmeyen = tus[tus["yan_dal"].isna()][COL_KONU].value_counts().head(40)
    print("\nEşleşmeyen en sık konular (dahiliye olup kaçan var mı kontrol et):")
    display(eslesmeyen.to_frame("soru_sayisi"))
else:
    oran = 0.0

In [ ]:
# Konu alanı branş bilgisi taşımıyorsa (eşleşme %5'in altındaysa) soru metninden TAHMİNİ etiketleme yapılır.
SORU_TERIMLERI = {
    "Kardiyoloji":        ["miyokard", "infarktus", "atriyal fibrilasyon", "kalp yetmezligi", "ekokardiyografi",
                           "angina", "perikardit", "endokardit", "hipertansiyon"],
    "Nefroloji":          ["kreatinin", "glomerulonefrit", "nefrotik", "diyaliz", "hiperkalemi", "hiponatremi",
                           "akut bobrek hasari", "kronik bobrek"],
    "Endokrinoloji":      ["hba1c", "insulin", "diyabetik", "tirotoksikoz", "hipotiroidi", "cushing", "addison",
                           "feokromositoma", "hiperparatiroidi", "akromegali"],
    "Gastroenteroloji":   ["siroz", "hepatit", "crohn", "ulseratif kolit", "gastrit", "pankreatit", "asit",
                           "ozofagus varis", "celiak", "colyak"],
    "Hematoloji":         ["demir eksikligi", "b12", "talasemi", "losemi", "lenfoma", "multipl miyelom",
                           "trombositopeni", "hemofili", "dic", "polisitemi"],
    "Romatoloji":         ["romatoid artrit", "lupus", "vaskulit", "behcet", "ankilozan", "gut", "sjogren",
                           "skleroderma", "ailevi akdeniz"],
    "Göğüs hastalıkları": ["koah", "astim", "pnomoni", "pulmoner emboli", "sarkoidoz", "plevral efuzyon",
                           "tuberkuloz", "bronsektazi"],
    "Enfeksiyon":         ["sepsis", "brusell", "menenjit", "hiv", "kkka", "sifiliz", "antibiyotik"],
}
HARIC_SORU = ["cocuk", "yenidogan", "bebek", "gebe", "gebelik", "operasyon", "cerrahi", "insizyon",
              "enzim eksikligi", "kofaktor", "reseptor alt tipi", "histolojik kesit"]

def sorudan_yan_dal(metin):
    m = sade(metin)
    if any(h in m for h in HARIC_SORU):
        return None
    puanlar = {dal: sum(t in m for t in terimler) for dal, terimler in SORU_TERIMLERI.items()}
    en_iyi = max(puanlar, key=puanlar.get)
    return en_iyi if puanlar[en_iyi] > 0 else None

if oran < 0.05:
    print("Konu alanı branş bilgisi taşımıyor görünüyor → soru metninden tahmini etiketleme yapılıyor.")
    bos = tus["yan_dal"].isna()
    tus.loc[bos, "yan_dal"] = tus.loc[bos, COL_SORU].map(sorudan_yan_dal)
    tus.loc[bos & tus["yan_dal"].notna(), "etiket_yontemi"] = "soru_metni_tahmini"
else:
    print("Konu alanı yeterli; soru metni yöntemi kullanılmadı.")

dahiliye = tus[tus["yan_dal"].notna()].copy()
print(f"\nDahiliye olarak etiketlenen TUS sorusu: {len(dahiliye):,}")
print(dahiliye["yan_dal"].value_counts().to_string())
print("\nEtiketleme yöntemi:")
print(dahiliye["etiket_yontemi"].value_counts().to_string())

## 6. Kalite filtreleri

In [ ]:
GORSEL_DESEN = re.compile(
    r"(sekil|gorsel|resim|fotograf|yukaridaki (tablo|grafik|goruntu|ekg)|asagidaki (tablo|grafik|goruntu|ekg)"
    r"|goruntuleme(de)? (gorulen|izlenen)|ekg(de|si)? (gorulen|verilen)|grafide (gorulen|izlenen))"
)
YANLIS_KALIP = re.compile(r"\b(hepsi|hicbiri|tumu|yukaridakilerin tumu)\b")

dahiliye["secenek_listesi"] = dahiliye[COL_SECENEK].map(secenek_listesi) if COL_SECENEK else [[]] * len(dahiliye)
dahiliye["dogru_harf"] = [cevap_harfi(c, s) for c, s in
                          zip(dahiliye[COL_CEVAP] if COL_CEVAP else [None] * len(dahiliye), dahiliye["secenek_listesi"])]
dahiliye["sade_soru"] = dahiliye[COL_SORU].map(sade)

dahiliye["f_gorsel"] = dahiliye["sade_soru"].str.contains(GORSEL_DESEN)
dahiliye["f_hepsi_hicbiri"] = dahiliye["secenek_listesi"].map(lambda s: any(YANLIS_KALIP.search(sade(x)) for x in s))
dahiliye["f_secenek_hatali"] = dahiliye["secenek_listesi"].map(lambda s: len(s) < 4 or len({sade(x) for x in s}) < len(s))
dahiliye["f_cevap_yok"] = dahiliye["dogru_harf"].isna()
dahiliye["f_cok_kisa"] = dahiliye["sade_soru"].str.len() < 20
dahiliye["f_tekrar"] = dahiliye.duplicated("sade_soru", keep="first")

filtreler = ["f_gorsel", "f_hepsi_hicbiri", "f_secenek_hatali", "f_cevap_yok", "f_cok_kisa", "f_tekrar"]
print("Filtrelere takılan soru sayıları (bir soru birden fazlasına takılabilir):")
for f in filtreler:
    print(f"   {f:<18}: {int(dahiliye[f].sum()):,}")

dahiliye["temiz"] = ~dahiliye[filtreler].any(axis=1)
temiz = dahiliye[dahiliye["temiz"]].copy()
print(f"\nTemizlik öncesi: {len(dahiliye):,}  →  temizlik sonrası: {len(temiz):,}")
print(temiz["yan_dal"].value_counts().to_string())

## 7. Test ve geliştirme setlerini ayırma, kaydetme

In [ ]:
TOHUM = 42
TEST_BOYUTU = 500
DEV_BOYUTU = 50

karisik = temiz.sample(frac=1, random_state=TOHUM)
test = karisik.iloc[:min(TEST_BOYUTU, len(karisik))]
dev = karisik.iloc[len(test):len(test) + DEV_BOYUTU]
havuz = karisik.iloc[len(test) + len(dev):]

CIKTI = "/content/tus_dahiliye_cikti" if os.path.isdir("/content") else "tus_dahiliye_cikti"
os.makedirs(CIKTI, exist_ok=True)

def disa_aktar(tablo, ad):
    kayitlar = []
    for i, r in tablo.iterrows():
        kayitlar.append({
            "id": f"tus_dah_{i}",
            "yan_dal": r["yan_dal"],
            "etiket_yontemi": r["etiket_yontemi"],
            "konu": r[COL_KONU] if COL_KONU else None,
            "soru": r[COL_SORU],
            "secenekler": r["secenek_listesi"],
            "dogru": r["dogru_harf"],
            "aciklama": r[COL_ACIKLAMA] if COL_ACIKLAMA else None,
            "kaynak_split": r["split"],
        })
    with open(f"{CIKTI}/{ad}.jsonl", "w", encoding="utf-8") as f:
        for k in kayitlar:
            f.write(json.dumps(k, ensure_ascii=False) + "\n")
    pd.DataFrame(kayitlar).to_csv(f"{CIKTI}/{ad}.csv", index=False, encoding="utf-8-sig")
    return len(kayitlar)

print("test :", disa_aktar(test, "tus_dahiliye_test"))
print("dev  :", disa_aktar(dev, "tus_dahiliye_dev"))
print("havuz:", disa_aktar(havuz, "tus_dahiliye_havuz"))
print("\nKaydedilen klasör:", CIKTI)

## 8. Rapor için özet (bu çıktıyı kopyalayıp Claude'a yapıştır)

In [ ]:
ozet = {
    "turkish_mmlu_toplam_satir": int(len(df)),
    "bolum_sayisi": int(df[COL_BOLUM].nunique()),
    "tus_bolum_adlari": [str(b) for b in TUS_BOLUMLERI],
    "tus_soru_sayisi": int(len(tus)),
    "tus_konu_sayisi": int(tus[COL_KONU].nunique()) if COL_KONU else None,
    "dahiliye_ham": int(len(dahiliye)),
    "dahiliye_etiket_yontemi": {str(k): int(v) for k, v in dahiliye["etiket_yontemi"].value_counts().items()},
    "filtreler": {f: int(dahiliye[f].sum()) for f in filtreler},
    "dahiliye_temiz": int(len(temiz)),
    "yan_dal_dagilimi_temiz": {str(k): int(v) for k, v in temiz["yan_dal"].value_counts().items()},
    "test": int(len(test)), "dev": int(len(dev)), "havuz": int(len(havuz)),
    "ornek_tus_konulari": [str(k) for k in (tus[COL_KONU].value_counts().head(25).index if COL_KONU else [])],
}
with open(f"{CIKTI}/ozet.json", "w", encoding="utf-8") as f:
    json.dump(ozet, f, ensure_ascii=False, indent=2)
print(json.dumps(ozet, ensure_ascii=False, indent=2))

## 9. Bonus: TUSGPT verisinde dahiliye tahmini ve test–eğitim çakışma kontrolü

TUSGPT veri seti giriş gerektirmez. Burada iki şeye bakılır:
* TUSGPT eğitim sorularının kabaca ne kadarı dahiliye terimleri içeriyor (yalnızca **ön tahmin**; asıl etiketleme LLM ile yapılacak).
* Test setindeki TUS sorularına **çok benzeyen** TUSGPT soruları var mı (kontaminasyon).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tg = load_dataset("turkerberkdonmez/TUSGPT-TR-Medical-Dataset-v1")
tg_train = tg["train"].to_pandas()
tg_train["yan_dal_tahmini"] = tg_train["instruction"].map(sorudan_yan_dal)
print(f"TUSGPT train: {len(tg_train):,} satır")
print(f"Terim eşleşmesiyle dahiliye görünen: {tg_train['yan_dal_tahmini'].notna().sum():,} "
      f"(%{100 * tg_train['yan_dal_tahmini'].notna().mean():.1f})")
print(tg_train["yan_dal_tahmini"].value_counts().to_string())

def cakisma_kontrolu(test_sorulari, egitim_sorulari, esik=0.80):
    vek = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2)
    vek.fit(list(map(sade, test_sorulari)) + list(map(sade, egitim_sorulari)))
    A = vek.transform(list(map(sade, test_sorulari)))
    B = vek.transform(list(map(sade, egitim_sorulari)))
    benzerlik = (A @ B.T).tocsr()
    sonuc = []
    for i in range(benzerlik.shape[0]):
        satir = benzerlik.getrow(i)
        if satir.nnz == 0:
            continue
        j = satir.indices[satir.data.argmax()]
        skor = satir.data.max()
        if skor >= esik:
            sonuc.append((i, int(j), float(skor)))
    return sonuc

supheli = cakisma_kontrolu(test[COL_SORU].tolist(), tg_train["instruction"].tolist())
print(f"\nTest setinde TUSGPT'ye çok benzeyen soru sayısı (benzerlik ≥ 0.80): {len(supheli)}")
for i, j, s in supheli[:10]:
    print(f"\n[{s:.2f}] TEST : {test[COL_SORU].iloc[i][:150]}")
    print(f"       TUSGPT: {tg_train['instruction'].iloc[j][:150]}")

In [ ]:
# Çıktıları tek dosya halinde indir
import shutil
arsiv = shutil.make_archive(CIKTI, "zip", CIKTI)
try:
    from google.colab import files
    files.download(arsiv)
except Exception:
    print("Arşiv:", arsiv)